# Nottingham Monthly Temperatures Dataset Integration

## 1. Orchestration boundary declaration and responsibility

This is the canonical, single human-facing Atlas dataset-integration
entrypoint for `nottem` (Project Spec S0251/S0252). It is the first real dataset
integration to exercise Atlas's completed native `univariate_forecasting`
capability (S0241-S0250) end to end: Atlas-local raw source verification,
deterministic raw-to-period translation, governed temporal semantic
authoring, `candidate-preparation-recipe.v2` expanding-window backtesting
authoring, reviewed frozen forecasting result/evaluation/training-policy
intents, capability-aware `execution_contract.v2` materialization,
runtime/public history-series contract projection, Atlas-native
`deterministic_seasonal_trend_ols` fixed-configuration forecasting training,
native forecasting evidence validation, `inference_bundle.v2` generation, a
controlled native forecast smoke check, release-candidate assembly,
Publisher Run materialization, and validated-run terminal handoff.

Like `notebooks/datasets/concrete-compressive-strength/dataset_integration.ipynb`
(Project Spec S0233), it remains an orchestrator: reusable generic
implementation logic lives in `pipeline/`, `publisher/`, and `runtime/`
modules, never in notebook cells. No notebook cell fits, tunes, selects, or
deserializes a model directly. The attached
`dataset-study-nottingham-monthly-temperatures` scientific reference project
is used only as implementation reference while authoring this notebook's
cells; at runtime this notebook never reads that project's path, never
loads its evidence, and never loads its model bytes. It also declares the single native Nottingham Predict View
(`nottem-temperature-forecast`) as presentation/intent metadata bound to
the active release -- `S0252` authorizes this declaration after `S0251`
explicitly deferred it. It stops unconditionally before publisher
promotion, registry activation, Predict View registry materialization,
Predict View customization seeding, or public-profile/visibility
activation.

In [ ]:
ORCHESTRATION_BOUNDARY = {
    "allowed": [
        "atlas_source_verification",
        "deterministic_raw_to_period_translation",
        "nottingham_temporal_semantic_authoring",
        "governed_temporal_preparation_authoring",
        "reviewed_forecasting_result_evaluation_training_policy_authoring",
        "univariate_forecasting_capability_resolution",
        "execution_contract_materialization",
        "runtime_public_history_series_contract_projection",
        "native_univariate_forecasting_training_run_materialization",
        "native_forecasting_metrics_visualization_evidence_validation",
        "inference_bundle_materialization",
        "controlled_native_forecast_smoke_check",
        "release_candidate_assembly",
        "publisher_structural_validation",
        "manifest_generation_when_structurally_permitted",
        "validated_run_terminal_handoff",
        "predict_view_declaration",
    ],
    "still_forbidden": [
        "external_scientific_project_read_at_runtime",
        "external_model_load",
        "external_evidence_ingestion",
        "network_dataset_download",
        "model_fitting_in_notebook",
        "model_retraining_in_notebook",
        "model_family_search",
        "hyperparameter_search",
        "seasonal_period_search",
        "feature_selection",
        "target_transformation",
        "model_deserialization_or_inference_execution_outside_controlled_smoke_check",
        "publisher_promotion",
        "registry_active_release_mutation",
        "predict_view_registry_materialization",
        "predict_view_customization_seeding",
        "public_visibility_or_profile_activation",
    ],
    "external_scientific_project_used_as_implementation_reference_only": True,
    "durable_absolute_external_path": False,
    "stops_before_promotion_registry_activation_and_public_prediction": True,
}
assert ORCHESTRATION_BOUNDARY["durable_absolute_external_path"] is False
assert ORCHESTRATION_BOUNDARY["stops_before_promotion_registry_activation_and_public_prediction"] is True
assert set(ORCHESTRATION_BOUNDARY["allowed"]).isdisjoint(ORCHESTRATION_BOUNDARY["still_forbidden"])

## 2. Atlas source identity / local raw input acquisition and verification

Uses Atlas's canonical local source boundary
(`data/raw/nottem/dataset.csv`, gitignored/uncommitted). The scientific
source is `datasets::nottem` (240 monthly observations, 1920-01 through
1939-12, degrees Fahrenheit) from Rdatasets. This notebook delegates
source acquisition to the generic Atlas-owned
`pipeline.dataset_acquisition.acquire_rdataset_csv` helper, passing only
the dataset/package identity -- never a raw URL. The helper materializes
the local raw file over Python-standard-library HTTPS when it is absent,
or reuses a valid existing local file without any network access. This
notebook itself contains no direct network/download implementation and
must never fall back to any external-study path, a second provider, or
an absolute filesystem path when acquisition fails.

In [ ]:
import math
from pathlib import Path

import pipeline.discovery_evidence as discovery_evidence_module
from pipeline.discovery_evidence import (
    resolve_repository_root, resolve_repository_path, load_dataset_csv, summarize_structure,
)
from pipeline.dataset_acquisition import acquire_rdataset_csv, DatasetAcquisitionError

repo_root = resolve_repository_root()


def resolve_imported_module_repo_root(module):
    return resolve_repository_root(module.__file__)


def assert_repository_import_root_coherence(module, expected_repo_root):
    imported_repo_root = resolve_imported_module_repo_root(module)
    if imported_repo_root != expected_repo_root:
        raise RuntimeError(
            "Atlas repository/import-root coherence check failed: this notebook "
            f"resolved repo_root={expected_repo_root}, but the imported "
            f"'{module.__name__}' module belongs to a different Atlas checkout "
            f"({imported_repo_root}). This usually means the active Python "
            "environment has a separate, non-editable install of the "
            "'pipeline' package (for example via `pip install .` instead of "
            "`pip install -e .`) shadowing this checkout. Fix the kernel's "
            "environment (reinstall in editable mode against this checkout, "
            "or remove the stray install) and restart the Jupyter kernel "
            "before re-running this notebook."
        )
    return imported_repo_root


assert_repository_import_root_coherence(discovery_evidence_module, repo_root)

dataset_slug = "nottem"
dataset_relative_path = "data/raw/nottem/dataset.csv"
scientific_source_row_count = 240
scientific_source_columns = ["time", "value"]

run_state = {"blocked": False, "reasons": []}


def record_block(code_, message, field=None):
    reason = {"code": code_, "message": message}
    if field is not None:
        reason["field"] = field
    run_state["blocked"] = True
    run_state["reasons"].append(reason)
    return reason


dataset_path = repo_root / dataset_relative_path

source_acquisition_result = {}
try:
    source_acquisition_result = acquire_rdataset_csv(
        dataset_name=dataset_slug,
        package="datasets",
        destination_relative_path=dataset_relative_path,
        expected_columns=scientific_source_columns,
        expected_row_count=scientific_source_row_count,
        repo_root=repo_root,
    )
except DatasetAcquisitionError as exc:
    record_block(
        "atlas_source_acquisition_failed",
        f"Atlas-owned source acquisition failed for {dataset_relative_path!r}: {exc}",
        "dataset_relative_path",
    )

In [ ]:
raw_rows = []
atlas_raw_structure = None

if not run_state["blocked"]:
    raw_rows = load_dataset_csv(dataset_path)
    atlas_raw_structure = summarize_structure(raw_rows)
    if atlas_raw_structure["ordered_columns"] != scientific_source_columns:
        record_block(
            "raw_source_columns_unexpected",
            f"expected raw columns exactly {scientific_source_columns!r}, found "
            f"{atlas_raw_structure['ordered_columns']!r}.",
            "ordered_columns",
        )
    if atlas_raw_structure["row_count"] != scientific_source_row_count:
        record_block(
            "raw_source_row_count_unexpected",
            f"expected {scientific_source_row_count} raw rows, found "
            f"{atlas_raw_structure['row_count']}.",
            "row_count",
        )

if not run_state["blocked"]:
    non_finite_row_positions = []
    for position, row in enumerate(raw_rows):
        try:
            raw_time_value = float(row["time"])
            raw_target_value = float(row["value"])
        except (TypeError, ValueError, KeyError):
            non_finite_row_positions.append(position)
            continue
        if not math.isfinite(raw_time_value) or not math.isfinite(raw_target_value):
            non_finite_row_positions.append(position)
    if non_finite_row_positions:
        record_block(
            "raw_source_non_finite_values",
            f"raw time/value columns must be finite numeric for every row; found "
            f"non-finite or non-numeric values at row positions {non_finite_row_positions!r}.",
            "time_or_value",
        )
    assert atlas_raw_structure["row_count"] == scientific_source_row_count

## 3. Reduced Atlas discovery evidence

Generic, dataset-agnostic discovery evidence generation
(`pipeline/discovery_evidence.py`), computed directly from the local Atlas
raw file, before any temporal preparation. No dataset-slug branch is
introduced anywhere in this call. This step's own row/column counts govern
the raw transport identity checked in Stage 2 above; it never observes
`period`/`temperature` names, which do not exist until Stage 5's governed
translation.

In [ ]:
from pipeline.discovery_evidence import materialize_discovery_evidence

discovery_evidence_relative_path = f"pipeline/evidence/{dataset_slug}/discovery-evidence.json"
discovery_evidence_seed = 42

atlas_discovery_evidence = {}
if not run_state["blocked"]:
    atlas_discovery_evidence = materialize_discovery_evidence(
        dataset_relative_path,
        discovery_evidence_relative_path,
        repo_root=repo_root,
        dataset_slug=dataset_slug,
        seed=discovery_evidence_seed,
    )
    assert atlas_discovery_evidence["dataset_metadata"]["row_count"] == scientific_source_row_count
    assert atlas_discovery_evidence["dataset_metadata"]["column_count"] == 2

## 4. Nottingham temporal semantic intent v4

Governed `dataset-semantic-intent.v4` authoring (Project Spec S0241):
encodes exactly the frozen Nottingham identity -- `temporal_index` field
`period`, `target` field `temperature`, `index_value_kind`
`calendar_period`, logical `frequency` `monthly`, and
`source_exogenous_predictors` fixed to `forbidden`. Unlike the Concrete
notebook's `dataset-semantic-intent.v3`, this document never defines a
continuous-regression `predicted_value` primary output, a scalar feature
list, forecast horizon, seasonal period, backtesting folds, or holdout
boundaries -- those remain owned by `candidate-preparation-recipe.v2` (Stage
5) and `execution_contract.v2` (Stage 8).

In [ ]:
import json

target_field_name = "temperature"
time_index_field_name = "period"
index_value_kind = "calendar_period"
frequency = "monthly"

authoring_generation_id = "nottem-authoring-v1"
generated_at = "2026-08-23T00:00:00+00:00"

field_role_decisions = [
    {
        "field_name": time_index_field_name,
        "role": "temporal_index",
        "include_in_features": False,
    },
    {
        "field_name": target_field_name,
        "role": "target",
        "include_in_features": False,
        "exclusion_reason": "Governed univariate forecasting target.",
    },
]

semantic_intent = {
    "schema_version": "dataset-semantic-intent.v4",
    "artifact_type": "dataset_semantic_intent",
    "dataset_identity": {
        "dataset_slug": dataset_slug,
        "dataset_logical_name": "Nottingham Monthly Temperatures",
    },
    "authoring_generation_id": authoring_generation_id,
    "governing_capability_profile": {
        "capability_profile_id": "univariate-predictive-forecasting",
        "capability_profile_version": "v1",
    },
    "field_role_decisions": field_role_decisions,
    "temporal_semantics": {
        "time_index_field_name": time_index_field_name,
        "index_value_kind": index_value_kind,
        "frequency": frequency,
        "source_exogenous_predictors": "forbidden",
    },
    "target_semantics": {
        "target_field_name": target_field_name,
        "task_type": "univariate_forecasting",
        "target_value_kind": "numeric",
        "forecasting_mode": "univariate",
        "is_final_training_configuration": False,
    },
    "authored_public_meaning": {
        "human_reviewed": True,
        "safe_projection_intent": (
            "Forecast Nottingham Castle monthly average air temperature twelve "
            "months ahead from its own governed monthly history."
        ),
    },
    "semantic_boundary_confirmations": {
        "observed_source_statistics_embedded": False,
        "scientific_conclusions_embedded": False,
        "training_outcome_embedded": False,
        "release_state_embedded": False,
        "model_bytes_embedded": False,
    },
    "generated_at": generated_at,
}
assert semantic_intent["temporal_semantics"]["source_exogenous_predictors"] == "forbidden"
assert semantic_intent["target_semantics"]["is_final_training_configuration"] is False

## 5. Governed temporal preparation / prepared series

Deterministically translates the Atlas-local raw `time,value` transport
table into the Atlas-owned `period,temperature` prepared series. The raw
`time` field is a fractional-year position from the R dataset
representation; each value must map unambiguously to the expected monthly
sequence position -- this step never rounds arbitrary malformed timestamps
into valid months, and never silently sorts or fills malformed source data.
The prepared series is a runtime artifact, never a repository source file.
Also authors `candidate-preparation-recipe.v2` (Project Spec S0242) with
the frozen development/holdout/expanding-window backtesting geometry from
Project Spec S0251 Section 1: development 1920-01 through 1938-12 (228
rows), a prospectively sealed final holdout 1939-01 through 1939-12 (12
rows), and nine expanding-window folds (initial 120 observations, 12-step
origin advance, horizon 12, 108 total validation forecasts) covering
1930 through 1938.

In [ ]:
EXPECTED_FIRST_YEAR = 1920
EXPECTED_LAST_PERIOD = "1939-12"


def period_label(position):
    year = EXPECTED_FIRST_YEAR + position // 12
    month = position % 12 + 1
    return f"{year:04d}-{month:02d}"


prepared_rows = []
if not run_state["blocked"]:
    translation_errors = []
    for position, row in enumerate(raw_rows):
        raw_time_value = float(row["time"])
        raw_target_value = float(row["value"])
        expected_fractional_year = EXPECTED_FIRST_YEAR + position / 12.0
        if abs(raw_time_value - expected_fractional_year) > 1e-6:
            translation_errors.append(
                f"row {position}: raw time value {raw_time_value!r} does not map "
                f"unambiguously to the expected monthly sequence position {position} "
                f"(expected approximately {expected_fractional_year!r})."
            )
            continue
        prepared_rows.append({"period": period_label(position), "temperature": raw_target_value})

    if translation_errors:
        for message in translation_errors:
            record_block("raw_to_period_translation_failed", message, "time")

if not run_state["blocked"]:
    prepared_periods = [row["period"] for row in prepared_rows]
    assert len(prepared_rows) == scientific_source_row_count
    assert prepared_periods[0] == "1920-01"
    assert prepared_periods[-1] == EXPECTED_LAST_PERIOD
    assert prepared_periods == sorted(set(prepared_periods))
    assert len(set(prepared_periods)) == len(prepared_periods)
    assert prepared_periods == [period_label(i) for i in range(scientific_source_row_count)]
    assert all(math.isfinite(row["temperature"]) for row in prepared_rows)

In [ ]:
import csv
import hashlib


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_governed_json(relative_path, payload):
    path = repo_root / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return {"path": relative_path, "sha256": sha256_file(path)}


prepared_data_relative_path = f"pipeline/prepared/{dataset_slug}/prepared-data.csv"
prepared_data_metadata_relative_path = f"pipeline/prepared/{dataset_slug}/prepared-data-metadata.json"

if not run_state["blocked"]:
    prepared_data_path = repo_root / prepared_data_relative_path
    prepared_data_path.parent.mkdir(parents=True, exist_ok=True)
    with prepared_data_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["period", "temperature"])
        writer.writeheader()
        writer.writerows(prepared_rows)

    prepared_data_metadata = {
        "schema_version": "prepared-data-metadata.v1",
        "dataset_identity": {"dataset_slug": dataset_slug},
        "prepared_candidate": {"produced": True, "reference": prepared_data_relative_path},
        "training_readiness": {"is_training_ready": True},
        "unresolved_review_items": [],
    }
    write_governed_json(prepared_data_metadata_relative_path, prepared_data_metadata)

In [ ]:
DEVELOPMENT_OBSERVATIONS = 228
SEALED_HOLDOUT_OBSERVATIONS = 12
FORECAST_HORIZON = 12
INITIAL_TRAINING_OBSERVATIONS = 120
ORIGIN_STEP_OBSERVATIONS = 12
FOLD_COUNT = 9
VALIDATION_FORECAST_COUNT = FOLD_COUNT * FORECAST_HORIZON
assert VALIDATION_FORECAST_COUNT == 108

fold_schedule = []
for fold_index in range(1, FOLD_COUNT + 1):
    training_observations = INITIAL_TRAINING_OBSERVATIONS + (fold_index - 1) * ORIGIN_STEP_OBSERVATIONS
    validation_start_position = training_observations
    validation_end_position = training_observations + FORECAST_HORIZON - 1
    fold_schedule.append({
        "fold_index": fold_index,
        "training_observations": training_observations,
        "forecast_origin": period_label(training_observations - 1),
        "validation_start": period_label(validation_start_position),
        "validation_end": period_label(validation_end_position),
        "validation_observations": FORECAST_HORIZON,
    })
assert fold_schedule[0]["validation_start"] == "1930-01"
assert fold_schedule[-1]["validation_end"] == "1938-12"
assert sum(entry["validation_observations"] for entry in fold_schedule) == VALIDATION_FORECAST_COUNT

preparation_recipe_relative_path = f"pipeline/authoring/{dataset_slug}/preparation-recipe.json"

if not run_state["blocked"]:
    temporal_integrity = {
        "strictly_increasing_index": prepared_periods == sorted(prepared_periods),
        "unique_index": len(set(prepared_periods)) == len(prepared_periods),
        "frequency_contiguous": prepared_periods == [period_label(i) for i in range(len(prepared_periods))],
        "target_missing_values_absent": all(row["temperature"] is not None for row in prepared_rows),
        "target_values_finite": all(math.isfinite(row["temperature"]) for row in prepared_rows),
    }
    if not all(temporal_integrity.values()):
        record_block(
            "prepared_series_temporal_integrity_failed",
            f"prepared series temporal_integrity confirmations are not all true: {temporal_integrity!r}.",
            "temporal_integrity",
        )

if not run_state["blocked"]:
    semantic_intent_ref = write_governed_json(
        f"pipeline/dataset-semantic-intent-instances/{dataset_slug}.json", semantic_intent
    )

    preparation_recipe = {
        "schema_version": "candidate-preparation-recipe.v2",
        "producer": "notebooks/datasets/nottem/dataset_integration.ipynb",
        "problem_type": "univariate_forecasting",
        "discovery_evidence_ref": {
            "path": discovery_evidence_relative_path,
            "schema_version": "dataset-discovery-evidence.v1",
        },
        "semantic_intent_ref": {
            "path": semantic_intent_ref["path"],
            "schema_version": "dataset-semantic-intent.v4",
            "sha256": semantic_intent_ref["sha256"],
        },
        "semantic_identity_mirror": {
            "time_index_field_name": time_index_field_name,
            "target_field_name": target_field_name,
            "index_value_kind": index_value_kind,
            "frequency": frequency,
        },
        "temporal_integrity": temporal_integrity,
        "forecast_horizon": FORECAST_HORIZON,
        "partitions": {
            "development": {
                "start_index_value": "1920-01",
                "end_index_value": "1938-12",
                "observation_count": DEVELOPMENT_OBSERVATIONS,
            },
            "sealed_final_holdout": {
                "start_index_value": "1939-01",
                "end_index_value": "1939-12",
                "observation_count": SEALED_HOLDOUT_OBSERVATIONS,
                "prospectively_sealed": True,
                "used_for_backtesting": False,
                "used_for_model_selection": False,
            },
        },
        "backtesting": {
            "mode": "expanding_window",
            "initial_training_observations": INITIAL_TRAINING_OBSERVATIONS,
            "forecast_horizon": FORECAST_HORIZON,
            "origin_step_observations": ORIGIN_STEP_OBSERVATIONS,
            "fold_count": FOLD_COUNT,
            "validation_targets_overlap": False,
        },
        "fold_schedule": fold_schedule,
        "leakage_controls": {
            "random_shuffle_performed": False,
            "future_targets_used_for_fold_fit": False,
            "final_holdout_used_for_backtesting": False,
            "final_holdout_used_for_model_selection": False,
            "validation_targets_fed_back_within_fold": False,
            "preprocessing_fit_on_validation_or_future": False,
        },
        "preparation_boundary_confirmations": {
            "model_training_performed": False,
            "release_publication_performed": False,
            "hidden_notebook_transformations": False,
        },
        "evidence_policy": {
            "raw_logs_prohibited": True,
            "raw_runtime_prohibited": True,
            "raw_api_payloads_prohibited": True,
            "secrets_prohibited": True,
            "private_source_paths_prohibited": True,
            "reduced_and_sanitized": True,
        },
        "generated_at": generated_at,
    }
    assert DEVELOPMENT_OBSERVATIONS + SEALED_HOLDOUT_OBSERVATIONS == scientific_source_row_count
    write_governed_json(preparation_recipe_relative_path, preparation_recipe)

## 6. Reviewed forecasting result, evaluation and fixed training policy intents

Uses the current S0243/S0244 authoring builders to freeze the
Nottingham-specific values from Project Spec S0251 Section 1: an approved
`forecast_series` / `ordered_forecast_points` result-semantics intent, an
approved MAE (primary) + RMSE + seasonal MASE(12) (secondary) evaluation
policy, and an approved `fixed_configuration`
`deterministic_seasonal_trend_ols` training policy (intercept, linear trend,
additive seasonal indicators, `seasonal_period=12`,
`reference_season_position=0`, `trend_origin=development_start`). These
values are frozen at the Nottingham-specific authoring level only -- they
are never promoted into generic forecasting defaults. All three intents
must be `approved` before execution-contract materialization.

In [ ]:
from pipeline.discovery_evidence import (
    build_univariate_forecasting_result_semantics_intent,
    build_univariate_forecasting_evaluation_policy_intent,
    build_univariate_forecasting_training_policy_intent,
    build_univariate_forecasting_history_input_policy_intent,
    build_univariate_forecasting_predictive_interaction_intent,
    build_dataset_modeling_intent,
    load_dataset_csv as load_prepared_csv,
    summarize_structure as summarize_prepared_structure,
    summarize_target_column as summarize_prepared_target_column,
)

nottingham_fixed_model_configuration = {
    "include_intercept": True,
    "linear_time_trend": True,
    "seasonal_effects": "additive_indicators",
    "seasonal_period": 12,
    "reference_season_position": 0,
    "trend_origin": "development_start",
}

nottingham_finalization_policy = {
    "backtesting_refit_each_fold": True,
    "final_fit_scope": "full_development",
    "freeze_before_final_holdout_open": True,
    "final_holdout_evaluation_count": 1,
    "final_holdout_used_for_adjustment": False,
    "final_holdout_used_for_model_selection": False,
    "no_retuning_after_final_holdout": True,
}

forecasting_result_semantics_intent = build_univariate_forecasting_result_semantics_intent(
    review_status="approved",
    problem_type="univariate_forecasting",
    primary_output="forecast_series",
    output_structure="ordered_forecast_points",
    forecast_value_kind="continuous_numeric",
    review_notes="Reviewed univariate forecast-series result semantics for Nottingham Monthly Temperatures.",
)

forecasting_evaluation_policy_intent = build_univariate_forecasting_evaluation_policy_intent(
    review_status="approved",
    primary_metric={"metric_id": "mae"},
    secondary_metrics=[
        {"metric_id": "rmse"},
        {"metric_id": "seasonal_mase", "seasonal_period": 12},
    ],
    review_notes="Reviewed governed evaluation policy: MAE primary, RMSE and seasonal MASE(12) secondary.",
)

forecasting_training_policy_intent = build_univariate_forecasting_training_policy_intent(
    review_status="approved",
    selection_mode="fixed_configuration",
    model_selection_performed=False,
    model_family="deterministic_seasonal_trend_ols",
    fixed_model_configuration=nottingham_fixed_model_configuration,
    finalization_policy=nottingham_finalization_policy,
    review_notes=(
        "Reviewed frozen deterministic_seasonal_trend_ols specification, replayed "
        "from the already-completed external scientific model selection -- no "
        "model search or tuning is performed by Atlas."
    ),
)

assert forecasting_result_semantics_intent["review_status"] == "approved"
assert forecasting_evaluation_policy_intent["review_status"] == "approved"
assert forecasting_training_policy_intent["review_status"] == "approved"

forecasting_history_input_policy_intent = build_univariate_forecasting_history_input_policy_intent(
    review_status="approved",
    minimum_observation_count=1,
    required_anchor={"presence": "required", "source": "development_end"},
    forecast_origin_source="last_validated_history_index",
    review_notes=(
        "Reviewed governed history-input policy: the currently supported "
        "generation requires the development-end anchor and advances forecast "
        "origin from the last validated supplied history index."
    ),
)

assert forecasting_history_input_policy_intent["review_status"] == "approved"

forecasting_predictive_interaction_intent = build_univariate_forecasting_predictive_interaction_intent(
    review_status="approved",
    history_target_values_affect_forecast=False,
    refit_on_input=False,
    model_parameters_updated_on_input=False,
    public_prediction_interaction_applicability="not_applicable",
    review_notes=(
        "Reviewed release-bound predictive-interaction semantics: the frozen "
        "deterministic_seasonal_trend_ols model uses supplied history to "
        "establish chronology and forecast origin only -- history target "
        "values do not alter frozen coefficients or the numeric forecast for "
        "a fixed temporal origin, the model is never refit or updated from "
        "inference-time input, and no public prediction-interaction "
        "capability is applicable for this release."
    ),
)

assert forecasting_predictive_interaction_intent["review_status"] == "approved"

In [ ]:
modeling_intent = {}
if not run_state["blocked"]:
    prepared_rows_for_authoring = load_prepared_csv(prepared_data_path)
    prepared_structure = summarize_prepared_structure(prepared_rows_for_authoring)
    prepared_target_observation = summarize_prepared_target_column(prepared_rows_for_authoring, target_field_name)

    modeling_intent = build_dataset_modeling_intent(
        dataset_slug=dataset_slug,
        dataset_source_ref=prepared_data_relative_path,
        authoring_notebook_ref="notebooks/datasets/nottem/dataset_integration.ipynb",
        columns=prepared_structure["ordered_columns"],
        target_column=target_field_name,
        task_type="univariate_forecasting",
        observed_labels=prepared_target_observation["observed_labels"],
        positive_label_candidate=None,
        observed_target_distribution=prepared_target_observation["observed_distribution"],
        identifier_columns=[],
        reduced_discovery_evidence_ref=discovery_evidence_relative_path,
        univariate_forecasting_result_semantics_intent=forecasting_result_semantics_intent,
        univariate_forecasting_evaluation_policy_intent=forecasting_evaluation_policy_intent,
        univariate_forecasting_training_policy_intent=forecasting_training_policy_intent,
        univariate_forecasting_history_input_policy_intent=forecasting_history_input_policy_intent,
        univariate_forecasting_predictive_interaction_intent=forecasting_predictive_interaction_intent,
        generated_at=generated_at,
    )

## 7. Univariate forecasting capability resolution

Resolves the governed `univariate-predictive-forecasting` capability
profile by explicit reference and requires `support_status ==
current_supported` (Project Spec S0250 flips the real committed profile to
`current_supported`) and `prediction_runtime.mode ==
single_model_univariate_forecasting`. No dataset slug may choose a
different generic branch.

In [ ]:
capability_profile_relative_path = "pipeline/capabilities/univariate-predictive-forecasting.v1.json"
capability_profile_path = repo_root / capability_profile_relative_path
capability_profile = json.loads(capability_profile_path.read_text(encoding="utf-8"))
assert capability_profile["schema_version"] == "capability-profile.v1"
assert capability_profile["capability_profile_id"] == "univariate-predictive-forecasting"
assert capability_profile["capability_profile_version"] == "v1"
if capability_profile["support_status"] != "current_supported":
    record_block(
        "capability_not_current_supported",
        f"univariate-predictive-forecasting support_status is "
        f"{capability_profile['support_status']!r}, not current_supported.",
        "support_status",
    )
if capability_profile["prediction_runtime"]["mode"] != "single_model_univariate_forecasting":
    record_block(
        "capability_prediction_runtime_mode_unexpected",
        f"prediction_runtime.mode is {capability_profile['prediction_runtime']['mode']!r}, "
        "not single_model_univariate_forecasting.",
        "prediction_runtime.mode",
    )

## 8. Execution contract v2 materialization

Materializes the official `execution_contract.v2` from the reviewed
forecasting intents (Stage 6), the semantic intent v4 (Stage 4), and the
governed `candidate-preparation-recipe.v2` (Stage 5) via
`pipeline.contract_derivation.materialize_execution_contract`. Cross-checks
target identity, time-index identity, frequency, forecast horizon,
preparation geometry, evaluation policy, training policy, and forecast-series
result semantics -- this notebook never hand-writes an execution contract.

In [ ]:
from pipeline.contract_derivation import materialize_execution_contract

execution_contract_relative_path = f"contracts/{dataset_slug}/execution-contract.json"
execution_contract_evidence_relative_path = f"contracts/{dataset_slug}/execution-contract-materialization-evidence.json"

execution_contract = {}
if not run_state["blocked"]:
    execution_contract_materialization = materialize_execution_contract(
        modeling_intent,
        atlas_discovery_evidence,
        execution_contract_relative_path,
        repo_root,
        preparation_recipe=preparation_recipe,
        evidence_output_relative_path=execution_contract_evidence_relative_path,
        semantic_intent=semantic_intent,
        generated_at=generated_at,
    )
    execution_contract = execution_contract_materialization["execution_contract"]

    assert execution_contract["contract_version"] == "execution_contract.v2"
    assert execution_contract["problem_type"] == "univariate_forecasting"
    assert execution_contract["target_column"] == target_field_name
    assert execution_contract["time_index_column"] == time_index_field_name
    assert execution_contract["frequency"] == frequency
    assert execution_contract["source_exogenous_predictors"] == "forbidden"
    assert execution_contract["forecast_horizon"] == FORECAST_HORIZON
    assert execution_contract["result_semantics"]["schema_version"] == "univariate-forecasting-result-semantics.v1"
    assert execution_contract["result_semantics"]["primary_output"] == "forecast_series"
    assert execution_contract["result_semantics"]["output_structure"] == "ordered_forecast_points"
    assert execution_contract["evaluation_policy"]["primary_metric"]["metric_id"] == "mae"
    assert {"rmse", "seasonal_mase"} <= {
        m["metric_id"] for m in execution_contract["evaluation_policy"]["secondary_metrics"]
    }
    assert execution_contract["training_policy"]["selection_mode"] == "fixed_configuration"
    assert execution_contract["training_policy"]["model_selection_performed"] is False
    assert execution_contract["training_policy"]["model_family"] == "deterministic_seasonal_trend_ols"
    assert execution_contract["training_policy"]["fixed_model_configuration"] == nottingham_fixed_model_configuration
    assert execution_contract["temporal_evaluation"]["backtesting_mode"] == "expanding_window"
    assert execution_contract["temporal_evaluation"]["fold_count"] == FOLD_COUNT
    assert execution_contract["temporal_evaluation"]["final_holdout_prospectively_sealed"] is True
    assert execution_contract["temporal_evaluation"]["random_shuffle_performed"] is False
    assert execution_contract["history_input_policy"]["schema_version"] == (
        "univariate-forecasting-history-input-policy.v1"
    )
    assert execution_contract["history_input_policy"]["minimum_observation_count"] == 1
    assert execution_contract["history_input_policy"]["required_anchor"] == {
        "presence": "required",
        "source": "development_end",
    }
    assert execution_contract["history_input_policy"]["forecast_origin_source"] == (
        "last_validated_history_index"
    )

    assert execution_contract["predictive_interaction_policy"]["schema_version"] == (
        "univariate-forecasting-predictive-interaction-policy.v1"
    )
    assert execution_contract["predictive_interaction_policy"]["history_target_values_affect_forecast"] is False
    assert execution_contract["predictive_interaction_policy"]["refit_on_input"] is False
    assert execution_contract["predictive_interaction_policy"]["model_parameters_updated_on_input"] is False
    assert execution_contract["predictive_interaction_policy"]["public_prediction_interaction_applicability"] == (
        "not_applicable"
    )

## 9. Runtime/public history-series contract projection

Invokes the current generic forecasting v2 projection
(`pipeline.derive_projections.derive`) with the explicit governing
`preparation_recipe_path`, producing `runtime-contract.json` (2.0.0) and
`public-contract.json` (2.0.0) with `payload_shape` /
`input_kind == history_series`, `period`/`temperature`/`monthly` identity,
`forecast_horizon=12`, and `caller_overridable=False`. Also materializes an Atlas-owned `dataset-context.json` that declares the
single native Nottingham Predict View (`nottem-temperature-forecast`) as
presentation/intent metadata bound to the active release -- `S0252`
authorizes this declaration after `S0251` explicitly deferred it. The
view carries no history-series/forecast validation semantics of its own;
canonical dataset contracts remain the sole source of runtime validation
truth (`contract_precedence`). Declaring the view in `dataset_context` is
not registry materialization, customization seeding, publisher promotion,
or public visibility activation -- those remain governed, separately
authorized, post-promotion operations.

In [ ]:
import pipeline.derive_projections as derive_projections_module
from pipeline.derive_projections import derive as derive_runtime_public_contracts, DerivationFailed

runtime_contract_relative_path = f"contracts/{dataset_slug}/runtime-contract.json"
public_contract_relative_path = f"contracts/{dataset_slug}/public-contract.json"
dataset_context_relative_path = f"contracts/{dataset_slug}/dataset-context.json"

if not run_state["blocked"]:
    assert_repository_import_root_coherence(derive_projections_module, repo_root)
    try:
        derive_runtime_public_contracts(
            repo_root / execution_contract_relative_path,
            repo_root / f"contracts/{dataset_slug}",
            repo_root=repo_root,
            preparation_recipe_path=repo_root / preparation_recipe_relative_path,
        )
    except DerivationFailed as exc:
        for message in exc.errors:
            record_block("runtime_public_projection_failed", message)

runtime_contract = {}
public_contract = {}
if not run_state["blocked"]:
    runtime_contract = json.loads((repo_root / runtime_contract_relative_path).read_text(encoding="utf-8"))
    public_contract = json.loads((repo_root / public_contract_relative_path).read_text(encoding="utf-8"))

    assert runtime_contract["schema_version"] == "2.0.0"
    assert runtime_contract["problem_type"] == "univariate_forecasting"
    assert runtime_contract["payload_shape"] == "history_series"
    assert runtime_contract["history_series"]["time_index_field_name"] == time_index_field_name
    assert runtime_contract["history_series"]["target_field_name"] == target_field_name
    assert runtime_contract["history_series"]["frequency"] == frequency
    assert runtime_contract["forecast"]["forecast_horizon"] == FORECAST_HORIZON
    assert runtime_contract["forecast"]["caller_overridable"] is False

    assert public_contract["schema_version"] == "2.0.0"
    assert public_contract["problem_type"] == "univariate_forecasting"
    assert public_contract["input_kind"] == "history_series"
    assert public_contract["history_series"]["frequency"] == frequency
    assert public_contract["forecast"]["forecast_horizon"] == FORECAST_HORIZON
    assert public_contract["forecast"]["horizon_user_editable"] is False

    # Project Spec S0266: the newly governed S0265 history_input_policy
    # must project bounded, safe public history guidance -- sourced from
    # the same governed execution_contract.history_input_policy and
    # preparation_recipe.partitions.development.end_index_value already
    # asserted above, never a Nottingham-specific hardcode and never a
    # copy of runtime-only/training terminology.
    assert public_contract["history_series"]["input_guidance"] == {
        "minimum_observation_count": execution_contract["history_input_policy"]["minimum_observation_count"],
        "required_anchor": {
            "display_value": preparation_recipe["partitions"]["development"]["end_index_value"],
        },
        "continuity": "consecutive_by_frequency",
    }
    assert public_contract["forecast"]["origin_behavior"] == "starts_after_last_history_observation"

    # Project Spec S0267: the current supported calendar_period + monthly
    # governed profile must also project the optional strict,
    # machine-actionable temporal_interaction guidance -- sourced from the
    # same governed execution_contract/preparation_recipe values already
    # asserted above, never a Nottingham-specific literal authored directly
    # into this assertion.
    assert public_contract["history_series"]["temporal_interaction"]["required_anchor"]["value"] == (
        preparation_recipe["partitions"]["development"]["end_index_value"]
    )
    assert public_contract["history_series"]["temporal_interaction"]["control_kind"] == "month"
    assert public_contract["history_series"]["temporal_interaction"]["required_anchor"]["inclusion"] == "required"
    assert public_contract["history_series"]["temporal_interaction"]["sequence"]["step_kind"] == "calendar_month"
    assert public_contract["history_series"]["temporal_interaction"]["sequence"]["continuity"] == "required"

    # Project Spec S0269: the newly governed release-bound
    # predictive_interaction_policy must project the bounded public
    # predictive-interaction facts -- sourced only from the same governed
    # execution_contract.predictive_interaction_policy already asserted
    # above, never a Nottingham-specific literal authored directly into
    # this assertion.
    assert public_contract["predictive_interaction"]["history_target_values"]["affect_forecast"] == (
        execution_contract["predictive_interaction_policy"]["history_target_values_affect_forecast"]
    )
    assert public_contract["predictive_interaction"]["public_prediction"]["applicability"] == (
        execution_contract["predictive_interaction_policy"]["public_prediction_interaction_applicability"]
    )

    # Project Spec S0252: canonical dataset-context shape carrying the
    # single native, dataset-owned Nottingham Predict View declaration.
    # Presentation/intent metadata only -- canonical contracts remain the
    # runtime validation and history-series contract source of truth
    # (contract_precedence below), never duplicated here. Project Spec
    # S0251 explicitly deferred this declaration; S0252 now authors it.
    nottingham_predict_view = {
        "schema_version": "1.0.0",
        "view_id": "nottem-temperature-forecast",
        "dataset_slug": dataset_slug,
        "display": {
            "title": "Nottingham Temperature Forecast",
            "summary": "Forecast future Nottingham monthly average air temperatures from the governed history series.",
            "description": "A univariate forecasting experience using the governed Nottingham Monthly Temperatures history-series input contract.",
            "tags": ["nottem", "forecasting", "univariate"],
        },
        "intent": {
            "prediction_goal": "Forecast future Nottingham monthly average air temperature from the canonical history-series dataset contract inputs.",
            "audience": "Users exploring Nottingham Monthly Temperatures univariate forecasting.",
            "usage_notes": "Use the canonical dataset contracts for history-series structure, temporal validation, fixed forecast horizon, and runtime validation.",
        },
        "binding": {
            "dataset_slug": dataset_slug,
            "release": {
                "mode": "active",
            },
        },
        "contract_precedence": {
            "canonical_contracts_are_source_of_truth": True,
            "view_metadata_defines_runtime_validation": False,
            "view_metadata_duplicates_contract": False,
        },
    }

    dataset_context = {
        "schema_version": "1.0.0",
        "dataset_slug": dataset_slug,
        "title": "Nottingham Monthly Temperatures",
        "description": (
            "Forecast Nottingham Castle monthly average air temperature twelve "
            "months ahead using Atlas's native univariate forecasting capability."
        ),
        "domain": "climate",
        "predict_views": [nottingham_predict_view],
    }
    write_governed_json(dataset_context_relative_path, dataset_context)

    # Validate the generated context against the canonical schema before
    # candidate assembly proceeds -- reuses this repository's existing
    # jsonschema mechanism (the same pattern pipeline.contract_derivation
    # already uses internally) rather than introducing a competing
    # validation framework, and blocks the run via the existing
    # record_block/run_state mechanism instead of raising.
    dataset_context_schema_path = repo_root / "contracts" / "dataset-context.schema.json"
    dataset_context_schema = json.loads(dataset_context_schema_path.read_text(encoding="utf-8"))
    try:
        import jsonschema
        dataset_context_schema_errors = sorted(
            jsonschema.Draft7Validator(dataset_context_schema).iter_errors(dataset_context),
            key=lambda e: list(e.path),
        )
        dataset_context_schema_error_messages = [e.message for e in dataset_context_schema_errors]
    except ImportError:
        dataset_context_schema_error_messages = (
            []
            if {"schema_version", "dataset_slug", "title", "description", "domain"} <= dataset_context.keys()
            else ["jsonschema library unavailable and required dataset-context fields are missing"]
        )
    for message in dataset_context_schema_error_messages:
        record_block("dataset_context_schema_invalid", message, "dataset_context")

    declared_predict_views = dataset_context.get("predict_views", [])
    if len(declared_predict_views) != 1:
        record_block(
            "dataset_context_predict_views_count_invalid",
            f"expected exactly one declared Predict View, found {len(declared_predict_views)}",
            "dataset_context.predict_views",
        )
    else:
        declared_view = declared_predict_views[0]
        if declared_view.get("view_id") != "nottem-temperature-forecast":
            record_block(
                "dataset_context_predict_view_id_invalid",
                f"expected view_id 'nottem-temperature-forecast', found {declared_view.get('view_id')!r}",
                "dataset_context.predict_views[0].view_id",
            )
        if declared_view.get("dataset_slug") != dataset_slug:
            record_block(
                "dataset_context_predict_view_dataset_slug_invalid",
                f"expected dataset_slug {dataset_slug!r}, found {declared_view.get('dataset_slug')!r}",
                "dataset_context.predict_views[0].dataset_slug",
            )
        declared_binding = declared_view.get("binding", {})
        if declared_binding.get("dataset_slug") != dataset_slug:
            record_block(
                "dataset_context_predict_view_binding_dataset_slug_invalid",
                f"expected binding.dataset_slug {dataset_slug!r}, found {declared_binding.get('dataset_slug')!r}",
                "dataset_context.predict_views[0].binding.dataset_slug",
            )
        if declared_binding.get("release", {}).get("mode") != "active":
            record_block(
                "dataset_context_predict_view_binding_release_mode_invalid",
                f"expected binding.release.mode 'active', found {declared_binding.get('release', {}).get('mode')!r}",
                "dataset_context.predict_views[0].binding.release.mode",
            )
        if declared_view.get("contract_precedence") != {
            "canonical_contracts_are_source_of_truth": True,
            "view_metadata_defines_runtime_validation": False,
            "view_metadata_duplicates_contract": False,
        }:
            record_block(
                "dataset_context_predict_view_contract_precedence_invalid",
                "declared contract_precedence does not match the required canonical-source-of-truth boundary",
                "dataset_context.predict_views[0].contract_precedence",
            )

## 10. Atlas-native forecasting training readiness

`pipeline.training.prepare_training_invocation_readiness` recognizes only
`execution_contract.v1` (Project Spec S0244's own `execution_contract.v2`
forecasting branch was never wired into that separate convenience bridge),
so reusing it here would misreport a valid forecasting contract as not
training-ready. This is a pre-existing gap in that generic bridge function,
not a reason to edit `pipeline/training.py` (out of scope) or to introduce
a dataset-specific conditional into generic code. This notebook instead
performs its own explicit, notebook-local readiness bridge -- checking the
real `execution_contract.v2` forecasting identity and the real prepared
series/preparation-recipe files on disk -- before calling the governed
`train_from_paths` entrypoint, which performs its own full, authoritative
input validation regardless.

In [ ]:
training_readiness = {"is_training_ready": False, "blocking_reasons": []}

if not run_state["blocked"]:
    reasons = []
    if execution_contract.get("contract_version") != "execution_contract.v2":
        reasons.append("execution_contract.contract_version is not execution_contract.v2")
    if execution_contract.get("problem_type") != "univariate_forecasting":
        reasons.append("execution_contract.problem_type is not univariate_forecasting")
    if not (repo_root / prepared_data_relative_path).is_file():
        reasons.append(f"prepared dataset does not exist: {prepared_data_relative_path}")
    if not (repo_root / preparation_recipe_relative_path).is_file():
        reasons.append(f"preparation recipe does not exist: {preparation_recipe_relative_path}")
    if not (repo_root / execution_contract_relative_path).is_file():
        reasons.append(f"execution contract does not exist: {execution_contract_relative_path}")

    training_readiness = {"is_training_ready": not reasons, "blocking_reasons": reasons}
    if not training_readiness["is_training_ready"]:
        for reason in training_readiness["blocking_reasons"]:
            record_block("training_not_ready", reason)

if not run_state["blocked"]:
    assert training_readiness["is_training_ready"] is True

## 11. Atlas-native forecasting training run materialization

Calls the governed native training entrypoint
`pipeline.training.train_from_paths(...)` directly, with an explicit
`preparation_recipe_path` -- never the older v1-only convenience
materializer (`materialize_training_run_from_prepared_metadata`) that omits
that argument. This notebook never calls `.fit(`, `fit_transform(`,
`GridSearchCV`, `RandomizedSearchCV`, `cross_validate`, or
`cross_val_score` on the model itself; all of that lives exclusively in
`pipeline/training.py`. `train_from_paths` raises `TrainingInputError`
(rather than returning a blocked-status dict) on any input failure, so this
stage catches that exception explicitly. This is the one real Atlas
`deterministic_seasonal_trend_ols` fixed-configuration forecasting training
run for `nottem`.

In [ ]:
from datetime import datetime, timezone

from pipeline import training as pipeline_training
from pipeline.training import train_from_paths, TrainingInputError

training_result = None
run_id = None

if not run_state["blocked"]:
    assert_repository_import_root_coherence(pipeline_training, repo_root)
    run_id = datetime.now(timezone.utc).strftime("train-%Y%m%dT%H%M%SZ")
    try:
        training_result = train_from_paths(
            repo_root / execution_contract_relative_path,
            repo_root / prepared_data_relative_path,
            preparation_recipe_path=repo_root / preparation_recipe_relative_path,
            dataset_slug=dataset_slug,
            run_id=run_id,
        )
    except TrainingInputError as exc:
        record_block("native_training_blocked", str(exc), getattr(exc, "field", None))

if not run_state["blocked"]:
    assert training_result.status == "trained"
    assert training_result.model_family == "deterministic_seasonal_trend_ols"
    assert training_result.task_type == "univariate_forecasting"
    assert training_result.feature_columns == []
    assert training_result.forecast_horizon == FORECAST_HORIZON

    training_run_materialization_result = {
        "status": "trained",
        "training_result": {
            "output_directory": training_result.output_directory,
            "training_parameter_record_path": training_result.training_parameter_record_path,
            "metrics_path": training_result.metrics_path,
            "serialized_model_path": training_result.serialized_model_path,
            "model_selection_evidence_path": training_result.model_selection_evidence_path,
        },
    }

    from pipeline import assemble_candidate

    release_id = assemble_candidate.derive_deterministic_release_id(run_id)

## 12. Native forecasting metrics / analytical evidence validation

Confirms the produced `training-parameter-record.v4`,
`training-metrics.v4`, and `analytical-visualizations.v4` artifacts declare
a real, sealed final-holdout evaluation, `model_selection_performed=False`,
and the frozen Nottingham model configuration, without recomputing any of
it. Compares the real, observed final-holdout MAE/RMSE/seasonal-MASE(12)
against the scientific reference values from Project Spec S0251 Section 1
under an explicit, narrow floating-point tolerance -- these reference
values are verification targets only, never hardcoded into the generated
evidence, and a real divergence blocks candidate assembly rather than
overwriting the observed evidence.

In [ ]:
training_parameter_record = {}
training_metrics = {}
analytical_visualizations = {}

if not run_state["blocked"]:
    training_parameter_record = json.loads(
        (repo_root / training_result.training_parameter_record_path).read_text(encoding="utf-8")
    )
    assert training_parameter_record["training_parameters"]["model_family"] == "deterministic_seasonal_trend_ols"
    assert training_parameter_record["training_parameters"]["selection_mode"] == "fixed_configuration"
    assert training_parameter_record["training_parameters"]["model_selection_performed"] is False

    training_metrics = json.loads((repo_root / training_result.metrics_path).read_text(encoding="utf-8"))
    assert training_metrics["schema_version"] == "training-metrics.v4"
    assert training_metrics["forecasting_evidence"]["problem_type"] == "univariate_forecasting"
    assert training_metrics["final_holdout_evaluation"]["evaluation_count"] == 1
    assert training_metrics["final_holdout_evaluation"]["observation_count"] == SEALED_HOLDOUT_OBSERVATIONS
    assert training_metrics["final_holdout_evaluation"]["model_frozen_before_open"] is True
    assert training_metrics["final_holdout_evaluation"]["used_for_adjustment"] is False
    assert training_metrics["final_holdout_evaluation"]["used_for_model_selection"] is False
    assert training_metrics["backtesting_evaluation"]["fold_count"] == FOLD_COUNT
    assert training_metrics["backtesting_evaluation"]["forecast_count"] == VALIDATION_FORECAST_COUNT

    analytical_visualizations = json.loads(
        (repo_root / training_result.analytical_visualizations_path).read_text(encoding="utf-8")
    )
    assert analytical_visualizations["schema_version"] == "analytical-visualizations.v4"
    assert analytical_visualizations["forecasting_evidence"]["problem_type"] == "univariate_forecasting"
    assert analytical_visualizations["dataset_statistics"]["development_observations"] == DEVELOPMENT_OBSERVATIONS
    assert analytical_visualizations["dataset_statistics"]["final_holdout_observations"] == SEALED_HOLDOUT_OBSERVATIONS

In [ ]:
SCIENTIFIC_CONTINUITY_REFERENCE_VALUES = {
    "mae": 1.526584,
    "rmse": 1.859967,
    "seasonal_mase": 0.555495,
}
SCIENTIFIC_CONTINUITY_ABSOLUTE_TOLERANCE = 0.05

if not run_state["blocked"]:
    observed_final_holdout_metrics = {
        entry["name"]: entry["value"] for entry in training_metrics["final_holdout_evaluation"]["metrics"]
    }
    for metric_name, reference_value in SCIENTIFIC_CONTINUITY_REFERENCE_VALUES.items():
        observed_value = observed_final_holdout_metrics.get(metric_name)
        if observed_value is None:
            record_block(
                "scientific_continuity_metric_missing",
                f"final-holdout metric {metric_name!r} is missing from real training evidence.",
                metric_name,
            )
            continue
        if abs(observed_value - reference_value) > SCIENTIFIC_CONTINUITY_ABSOLUTE_TOLERANCE:
            record_block(
                "scientific_continuity_mismatch",
                f"observed final-holdout {metric_name}={observed_value!r} diverges from the "
                f"scientific reference value {reference_value!r} by more than the explicit "
                f"tolerance {SCIENTIFIC_CONTINUITY_ABSOLUTE_TOLERANCE!r}.",
                metric_name,
            )
    # The reference values above are comparison targets only -- this
    # notebook never writes them into training_metrics/training_parameter_record.
    assert "scientific_continuity" not in training_metrics

## 13. Governed inference-bundle generation and native forecast smoke check

Materializes `inference_bundle.v2` from the training run's own governed
artifacts through
`generate_inference_bundle.materialize_governed_inference_bundle` -- the
same generic entrypoint used by every other dataset, dispatched to its
forecasting branch purely by `execution_contract.v2`'s own declared
`problem_type`. Unlike the v1 tabular branch (Concrete), the forecasting
branch derives its own provisional `release_context.release_id` internally
from the training run id and ignores any `release_id` passed in; the real
`release_id` used for candidate assembly below (Stage 14) is the
independent, separately allocated `assemble_candidate.derive_deterministic_release_id(run_id)`
computed in Stage 11.

In [ ]:
from pipeline import generate_inference_bundle

inference_bundle_relative_path = f"pipeline/inference-bundles/{dataset_slug}/inference-bundle.json"
inference_bundle_result = {}

if not run_state["blocked"]:
    inference_bundle_result = generate_inference_bundle.materialize_governed_inference_bundle(
        training_run_materialization_result=training_run_materialization_result,
        execution_contract_path=repo_root / execution_contract_relative_path,
        runtime_contract_path=repo_root / runtime_contract_relative_path,
        public_contract_path=repo_root / public_contract_relative_path,
        dataset_context_path=repo_root / dataset_context_relative_path,
        output_path=repo_root / inference_bundle_relative_path,
        repo_root=repo_root,
        dataset_slug=dataset_slug,
        execution_contract_ref=execution_contract_relative_path,
        runtime_contract_ref=runtime_contract_relative_path,
        public_contract_ref=public_contract_relative_path,
        dataset_context_ref=dataset_context_relative_path,
        model_package_reference="models/model.pkl",
    )
    if inference_bundle_result["status"] != "generated":
        for reason in inference_bundle_result["blocking_reasons"]:
            record_block("inference_bundle_blocked", reason)

In [ ]:
inference_bundle = {}
if not run_state["blocked"]:
    inference_bundle = json.loads((repo_root / inference_bundle_relative_path).read_text(encoding="utf-8"))
    assert inference_bundle["contract_version"] == "inference_bundle.v2"
    assert inference_bundle["model_provenance_origin"] == "atlas_internal_training"
    assert inference_bundle["frozen_model"]["model_family"] == "deterministic_seasonal_trend_ols"
    assert inference_bundle["frozen_model"]["state"] == "frozen"
    assert inference_bundle["frozen_model"]["fixed_model_configuration"] == nottingham_fixed_model_configuration
    assert inference_bundle["frozen_model"]["temporal_identity"]["target_column"] == target_field_name
    assert inference_bundle["frozen_model"]["temporal_identity"]["time_index_column"] == time_index_field_name
    assert inference_bundle["frozen_model"]["temporal_identity"]["frequency"] == frequency
    assert inference_bundle["frozen_model"]["temporal_identity"]["forecast_horizon"] == FORECAST_HORIZON
    assert inference_bundle["frozen_model"]["temporal_identity"]["source_exogenous_predictors"] == "forbidden"
    assert inference_bundle["output_schema"]["result_schema_version"] == "univariate-forecasting-result.v1"
    assert inference_bundle["result_semantics"]["primary_output"] == "forecast_series"
    assert "feature_order" not in inference_bundle
    assert "preprocessing" not in inference_bundle
    assert "external_model_evidence" not in inference_bundle

### Native forecast smoke check

Performs one controlled, in-process invocation of the current generic
native forecasting runtime execution path
(`runtime.inference.execute_prediction`), using only a scratch, non-
repository `tempfile.TemporaryDirectory()` "release root" populated with a
copy of the real bundle/model artifacts just produced -- never a real
`releases/` directory, never the promotion/registry machinery, and never a
public API HTTP endpoint. Requires exactly twelve ordered, strictly
increasing, monthly future periods and finite numeric forecast values with
`problem_type == univariate_forecasting`.

In [ ]:
import shutil
import tempfile

import runtime.inference as runtime_inference_module
import sys

API_ROOT = repo_root / "api"
if str(API_ROOT) not in sys.path:
    sys.path.insert(0, str(API_ROOT))
from payload_validator import validate_and_normalize_payload

native_forecast_smoke_result = None

if not run_state["blocked"]:
    assert_repository_import_root_coherence(runtime_inference_module, repo_root)

    with tempfile.TemporaryDirectory(prefix="nottem-forecast-smoke-check-") as scratch_release_dir:
        scratch_release_root = Path(scratch_release_dir)

        scratch_bundle_path = scratch_release_root / "predictions" / "bundle.json"
        scratch_bundle_path.parent.mkdir(parents=True, exist_ok=True)
        scratch_bundle_path.write_text(json.dumps(inference_bundle), encoding="utf-8")

        scratch_model_path = scratch_release_root / "models" / "model.pkl"
        scratch_model_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(repo_root / training_result.serialized_model_path, scratch_model_path)

        smoke_history_rows = [
            {"period": row["period"], "temperature": row["temperature"]}
            for row in prepared_rows[:DEVELOPMENT_OBSERVATIONS]
        ]
        payload_report = validate_and_normalize_payload(
            {"history": smoke_history_rows}, runtime_contract,
        )
        if payload_report.failures:
            record_block(
                "native_forecast_smoke_check_payload_invalid",
                f"synthetic local history series failed runtime payload validation: "
                f"{[f.error_code for f in payload_report.failures]!r}.",
            )
        else:
            prediction_outcome = runtime_inference_module.execute_prediction(
                active_release={"release_root": str(scratch_release_root)},
                validated_payload=payload_report.normalized_payload,
                manifest={"inference_bundle": "predictions/bundle.json"},
                bundle_loader=lambda path: json.loads(path.read_text(encoding="utf-8")),
                loader_strategies={
                    runtime_inference_module.JOBLIB_SKLEARN_FORECASTING_ADAPTER_STRATEGY:
                        runtime_inference_module.load_joblib_sklearn_model,
                },
                runtime_contract=runtime_contract,
            )
            native_forecast_smoke_result = prediction_outcome["result"]
            runtime_inference_module.validate_univariate_forecasting_result(native_forecast_smoke_result)

if not run_state["blocked"]:
    assert native_forecast_smoke_result["problem_type"] == "univariate_forecasting"
    assert native_forecast_smoke_result["frequency"] == frequency
    smoke_forecast_points = native_forecast_smoke_result["forecast_points"]
    assert len(smoke_forecast_points) == FORECAST_HORIZON
    smoke_forecast_periods = [point["future_time_index"] for point in smoke_forecast_points]
    assert smoke_forecast_periods == sorted(smoke_forecast_periods)
    assert len(set(smoke_forecast_periods)) == len(smoke_forecast_periods)
    assert all(math.isfinite(point["forecast"]) for point in smoke_forecast_points)
    assert [point["horizon_step"] for point in smoke_forecast_points] == list(range(1, FORECAST_HORIZON + 1))

## 14. Release-candidate role readiness and assembly

`pipeline.training.py` never produces a `model_card_path` for forecasting
(`TrainingResult.model_card_path` is `None` for the `univariate_forecasting`
branch), while release-candidate assembly still requires the `model_card`
role. Unlike `predictive_bundle`/`training_metrics`/`visualizations`,
`publisher/validate.py` carries no dedicated forecasting compatibility
check for `model_card` -- it only requires valid, parseable JSON with an
internally consistent `model_id` when one is declared. A truthful, reduced
`model-card.v1` can therefore be authored directly here, from the
Atlas-owned training/evaluation evidence already validated above, without
fabricating a fake scalar-feature list, a random/stratified split policy,
or a classification/regression task identity. If a future generic
model-card contract change ever required tabular semantics that forecasting
cannot truthfully satisfy, this stage is designed to record a deterministic
block here and never reach candidate assembly.

In [ ]:
model_card_relative_path = f"{training_result.output_directory}model-card.json" if not run_state["blocked"] else None
model_card_role_truthfully_representable = True

if not run_state["blocked"]:
    assert training_result.model_card_path is None
    if model_card_role_truthfully_representable:
        model_card = {
            "role": "model_card",
            "schema_version": "model-card.v1",
            "dataset_identity": {"dataset_slug": dataset_slug},
            "release_identity": {"release_id": release_id},
            "problem_type": "univariate_forecasting",
            "prediction_target": target_field_name,
            "input_features": [],
            "model_family": "deterministic_seasonal_trend_ols",
            "fixed_model_configuration": nottingham_fixed_model_configuration,
            "evaluation": {
                "partition_role": "final_holdout",
                "metrics": training_metrics["final_holdout_evaluation"]["metrics"],
            },
            "notes": (
                "Univariate forecasting model card. Input is a governed "
                "period|temperature history series, not a tabular feature "
                "vector -- input_features is intentionally empty; this is not "
                "a missing-feature omission. No random/stratified split policy "
                "or classification/continuous-regression scalar-target "
                "semantics apply to this forecasting model."
            ),
            "generated_at": generated_at,
        }
        write_governed_json(model_card_relative_path, model_card)
        for forbidden_key in ("feature_columns", "split_policy", "classes", "positive_class", "threshold", "predicted_value"):
            assert forbidden_key not in model_card
    else:
        record_block(
            "model_card_role_not_truthfully_representable",
            "the current candidate/publisher model-card boundary cannot accept a "
            "truthful forecasting model card without generic production-code changes.",
        )

In [ ]:
from pipeline import assemble_candidate

candidate_artifact_references = {}
candidate_handoff_readiness = {"is_release_candidate_input_ready": False, "not_ready_roles": []}

if not run_state["blocked"]:
    candidate_artifact_references = {
        "discovery_evidence": discovery_evidence_relative_path,
        "execution_contract": execution_contract_relative_path,
        "runtime_contract": runtime_contract_relative_path,
        "public_contract": public_contract_relative_path,
        "preparation_recipe": preparation_recipe_relative_path,
        "prepared_data_metadata": prepared_data_metadata_relative_path,
        "training_parameter_record": training_result.training_parameter_record_path,
        "model_artifact": training_result.serialized_model_path,
        "training_metrics": training_result.metrics_path,
        "model_card": model_card_relative_path,
        "public_context": dataset_context_relative_path,
        "visualizations": training_result.analytical_visualizations_path,
        "inference_bundle": inference_bundle_relative_path,
    }

    candidate_handoff_readiness = assemble_candidate.build_release_candidate_handoff_readiness(
        candidate_artifact_references, repo_root=repo_root
    )
    if not candidate_handoff_readiness["is_release_candidate_input_ready"]:
        for unready_role in candidate_handoff_readiness["not_ready_roles"]:
            record_block(
                "candidate_role_unavailable",
                f"release-candidate handoff role is not ready: {unready_role}",
                unready_role,
            )

In [ ]:
assembly_result = None

if not run_state["blocked"]:
    candidate_input = assemble_candidate.build_release_candidate_input(
        dataset_slug=dataset_slug,
        release_id=release_id,
        source_run_id=run_id,
        artifact_references=candidate_artifact_references,
        repo_root=repo_root,
        release_version="1.0.0-rc.1",
        dataset_title="Nottingham Monthly Temperatures",
    )

    candidate_output_dir = repo_root / "releases" / "candidates"
    assembly_result = assemble_candidate.assemble_release_candidate(
        candidate_input, candidate_output_dir, repo_root=repo_root,
    )
    if assembly_result["status"] != "accepted":
        record_block("candidate_assembly_rejected", assembly_result.get("reason", "assembly rejected"))

## 15. Publisher Run materialization + manifest

Calls the dataset-generic `publisher.validate.materialize_validation_run`
(Project Spec S0217) with the explicit `assembly_result` returned by
candidate assembly above -- never a `releases/candidates` scan and never a
dataset-slug dispatch branch. This materializes exactly one Publisher Run
(`publisher.validate.run`, writing `validation-result.json`), and, only
when the run is structurally accepted, its manifest
(`publisher.manifest.run`, writing `manifest.json`) in the same run
directory. No promotion and no registry mutation occur here. The notebook
blocks unless the run is materialized, structurally accepted, and its
manifest is generated.

In [ ]:
from publisher import validate as publisher_validate

publisher_materialization_result = None
publisher_run_id = None
publisher_run_dir_relative_path = None
publisher_validation_outcome = None
publisher_manifest_relative_path = None

if not run_state["blocked"]:
    publisher_materialization_result = publisher_validate.materialize_validation_run(
        assembly_result, repo_root=repo_root,
    )

    if publisher_materialization_result["materialization_status"] != "materialized":
        record_block(
            "publisher_run_materialization_blocked",
            publisher_materialization_result.get("message")
            or f"publisher run materialization blocked: {publisher_materialization_result.get('reason_code')}",
        )
    else:
        publisher_run_id = publisher_materialization_result["run_id"]
        publisher_run_dir_relative_path = publisher_materialization_result["run_dir"]
        publisher_validation_outcome = publisher_materialization_result["validation_outcome"]
        publisher_manifest_relative_path = publisher_materialization_result["manifest_path"]
        publisher_run_dir = repo_root / publisher_run_dir_relative_path

        if publisher_validation_outcome != "accepted":
            record_block(
                "publisher_structural_validation_rejected",
                "publisher.validate.run did not accept the assembled candidate.",
            )
        elif not publisher_materialization_result["manifest_generated"]:
            record_block(
                "publisher_manifest_not_generated",
                publisher_materialization_result.get("manifest_error")
                or "manifest was not generated for an accepted Publisher Run.",
            )
        elif not publisher_run_dir.is_dir():
            record_block("publisher_run_directory_missing", "Publisher Run directory does not exist.")
        elif not (publisher_run_dir / "validation-result.json").is_file():
            record_block(
                "publisher_validation_result_missing",
                "validation-result.json is missing from the Publisher Run directory.",
            )
        elif not (publisher_run_dir / "manifest.json").is_file():
            record_block(
                "publisher_manifest_file_missing",
                "manifest.json is missing from the Publisher Run directory.",
            )

## 16. Validated-run terminal result

Exactly one explicit validated-run terminal outcome is materialized through
`pipeline.validated_run.materialize_validated_run_terminal_result`, using
`model_source_mode = atlas_internal_training` -- the same internal-training
semantics used by every other native Atlas training run. This notebook
never reimplements promotion-eligibility logic; the generic terminal
producer owns the eligibility/hash/schema decisions. Blocked lower stages
propagate their concrete reason codes/messages into this terminal outcome.
The schema-valid result is persisted through the existing narrow JSON-write
helper into the same already-governed Publisher Run directory materialized
in Stage 15 above -- never `pipeline/training-runs/` or
`releases/candidates/`. This step only runs when a Publisher Run directory
actually exists above.

In [ ]:
from pipeline import validated_run


def durable_ref(path_value):
    if path_value is None:
        return None
    return {"path": path_value, "sha256": sha256_file(repo_root / path_value)}


validated_run_terminal_result = None
terminal_result_ref = None

if publisher_run_dir_relative_path is not None:
    release_candidate_json_relative_path = None
    if assembly_result is not None and assembly_result.get("status") == "accepted":
        release_candidate_json_relative_path = (
            f"{assembly_result['candidate_dir']}/release-candidate.json".replace(str(repo_root) + "/", "")
        )

    durable_references = {
        "materialization_result": None,
        "inference_bundle": durable_ref(
            inference_bundle_relative_path
            if inference_bundle_result and inference_bundle_result.get("status") == "generated"
            else None
        ),
        "release_candidate": durable_ref(release_candidate_json_relative_path),
        "publisher_validation_result": durable_ref(f"{publisher_run_dir_relative_path}/validation-result.json"),
        "manifest": durable_ref(publisher_manifest_relative_path),
        "operational_readiness_source": None,
    }

    structural_validation = (
        {"validation_outcome": publisher_validation_outcome}
        if publisher_validation_outcome is not None
        else None
    )

    manifest_outcome = (
        {
            "manifest_generated": publisher_materialization_result["manifest_generated"],
            "manifest_path": publisher_manifest_relative_path,
        }
        if publisher_materialization_result is not None
        else None
    )

    # Atlas-native internal training never carries an external
    # operational-readiness profile.
    operational_readiness = {
        "operational_validity": "not_applicable",
        "operational_threshold": {"status": "not_applicable", "value": None},
        "operational_prediction_available": False,
    }

    terminal_status = "blocked" if run_state["blocked"] else "completed"
    terminal_reasons = run_state["reasons"] if run_state["blocked"] else None

    validated_run_terminal_result = validated_run.materialize_validated_run_terminal_result(
        run_id=run_id,
        dataset_slug=dataset_slug,
        model_source_mode="atlas_internal_training",
        status=terminal_status,
        durable_references=durable_references,
        structural_validation=structural_validation,
        manifest_outcome=manifest_outcome,
        operational_readiness=operational_readiness,
        reasons=terminal_reasons,
        repo_root=repo_root,
    )

    terminal_result_relative_path = f"{publisher_run_dir_relative_path}/validated-run-terminal-result.json"
    terminal_result_ref = write_governed_json(terminal_result_relative_path, validated_run_terminal_result)

    assert validated_run_terminal_result["status"] in ("completed", "blocked", "failed")
    assert validated_run_terminal_result["promotion_eligibility"] in (True, False)
    if validated_run_terminal_result["status"] == "completed":
        assert validated_run_terminal_result["promotion_eligibility"] is True

## 17. Explicit stop before promotion / registry / public-profile / Predict View activation

This notebook stops here. It declares the single native Nottingham
Predict View above (Stage 9), but it never calls `publisher.promote.run`,
`registry.update.run`, or any public visibility/profile-activation
entrypoint, and it performs no Predict View registry materialization, no
Predict View customization seeding, no Nottingham-specific Inference Form
default, and no Admin/public Nottingham content -- those remain governed,
post-promotion operations performed by `registry.update` and the existing
generic Admin customization flow after a promoted release exists.
Publisher Run materialization, manifest generation, and the validated
terminal handoff are already durably persisted above (Stage 15 / Stage 16).

In [ ]:
orchestration_summary = {
    "dataset_slug": dataset_slug,
    "run_blocked": run_state["blocked"],
    "blocking_reasons": run_state["reasons"],
    "publisher_run_id": publisher_run_id,
    "publisher_run_dir": publisher_run_dir_relative_path,
    "terminal_status": validated_run_terminal_result["status"] if validated_run_terminal_result else None,
    "promotion_eligibility": (
        validated_run_terminal_result["promotion_eligibility"] if validated_run_terminal_result else False
    ),
    "stops_before_promotion_registry_activation_and_public_prediction": True,
    "promotion_performed": False,
    "registry_activation_performed": False,
    "public_visibility_or_profile_activation_performed": False,
    "predict_view_declared": True,
    "predict_view_registry_materialized": False,
    "predict_view_customization_seeded": False,
    "source_reference": source_acquisition_result.get("source_reference"),
    "source_materialization_status": source_acquisition_result.get("materialization_status"),
    "source_relative_path": source_acquisition_result.get("relative_path"),
}
orchestration_summary